# Notebook 17 - Dual-View Trained Adapter Gate

Checks whether paired full-image plus router/SAM ROI adapter training is justified by Notebook 16 results before starting an expensive second-phase training run.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            repo_root = candidate.resolve()
            os.chdir(repo_root)
            try:
                subprocess.run(['git', 'pull', '--ff-only'], check=True)
            except Exception as exc:
                print(f'[BOOT] git pull skipped/failed: {exc}', flush=True)
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    os.chdir(CLONE_TARGET)
    return CLONE_TARGET

ROOT = _ensure_aads_repo_on_path()
from scripts.colab_notebook_bootstrap_helpers import setup_notebook_environment
ROOT = setup_notebook_environment(install_requirements=True, print_tokens=True)
print(f'[BOOT] ROOT={ROOT}')


In [ ]:
from pathlib import Path
import importlib
import scripts.colab_roi_ablation as roi_ablation

roi_ablation = importlib.reload(roi_ablation)
commit_and_push_ablation_results = roi_ablation.commit_and_push_ablation_results
evaluate_dual_view_training_gate = roi_ablation.evaluate_dual_view_training_gate

ABLATION_NAME = 'dual_view_trained_adapter'
OUTPUT_DIR = ROOT / 'docs' / 'ablation_results' / ABLATION_NAME
BASELINE_REPORT = ROOT / 'docs' / 'ablation_results' / 'full_image_baseline' / 'report.json'
DUAL_VIEW_REPORT = ROOT / 'docs' / 'ablation_results' / 'dual_view_inference' / 'report.json'
MIN_ACCURACY_DELTA = 0.0
MIN_MACRO_F1_DELTA = 0.0

print(f'[CONFIG] ablation={ABLATION_NAME} output={OUTPUT_DIR}')


In [ ]:
gate = evaluate_dual_view_training_gate(
    repo_root=ROOT,
    output_dir=OUTPUT_DIR,
    baseline_report=BASELINE_REPORT,
    dual_view_report=DUAL_VIEW_REPORT,
    min_accuracy_delta=MIN_ACCURACY_DELTA,
    min_macro_f1_delta=MIN_MACRO_F1_DELTA,
)
git_result = commit_and_push_ablation_results(
    OUTPUT_DIR,
    repo_root=ROOT,
    message='Add dual-view trained adapter gate result',
)
{'gate': gate, 'git': git_result}
